# Gradient Descent vs Newton's Method

When we call `loss.backward()` in PyTorch, each parameter gets a `.grad` — a partial derivative computed by holding all other weights constant. But all weights update simultaneously. So we're breaking our own assumption.

Newton's method doesn't make that assumption. It uses the Hessian to account for how changing one weight affects the optimal update for another.

Let's train a linear regression model both ways and see the difference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## The Data

We generate synthetic data with two **correlated** features. This is key — correlated features mean the weights are entangled. Changing one weight changes what the optimal value of the other should be.

This is the $xy$ cross-term from the blog post, but in a real regression setting.

In [ ]:
n_samples = 200

# Two features with correlation 0.8
cov = [[1, 0.8],
       [0.8, 1]]
X = np.random.multivariate_normal([0, 0], cov, n_samples)

# True relationship: y = 3*x1 - 2*x2 + noise
w_true = np.array([3.0, -2.0])
y = X @ w_true + np.random.randn(n_samples) * 1.5

print(f"X shape: {X.shape}")  # (200, 2)
print(f"y shape: {y.shape}")  # (200,)
print(f"\nFeature correlation: {np.corrcoef(X.T)[0, 1]:.3f}")

## The Loss

Mean Squared Error. We want to minimize:

$$L(\mathbf{w}) = \frac{1}{n}\|X\mathbf{w} - \mathbf{y}\|^2$$

In [ ]:
def mse_loss(X, y, w):
    residuals = X @ w - y
    return np.mean(residuals ** 2)

print(f"Initial loss (w = [0, 0]): {mse_loss(X, y, np.zeros(2)):.2f}")

## The Hessian

Before we train anything, let's look at the Hessian of this loss. For MSE, it's $H = \frac{2}{n}X^TX$.

The off-diagonal entry is the cross-term — it tells us how entangled the two weights are. Gradient descent ignores it. Newton's method uses it.

In [ ]:
n = X.shape[0]
H = (2 / n) * X.T @ X

print(f"Hessian:")
print(f"  [{H[0,0]:.4f}  {H[0,1]:.4f}]")
print(f"  [{H[1,0]:.4f}  {H[1,1]:.4f}]")
print(f"")
print(f"Cross-term: {H[0,1]:.4f}")
print(f"")
print(f"This is NOT zero. The weights are entangled.")
print(f"Gradient descent ignores this. Newton's method uses it.")

## Gradient Descent

The gradient of MSE: $\nabla L = \frac{2}{n}X^T(X\mathbf{w} - \mathbf{y})$

Each component of this gradient is a partial derivative — computed by holding the other weight constant. We update both weights simultaneously using these values.

In [ ]:
def gradient_descent(X, y, lr, n_steps):
    n = X.shape[0]
    w = np.zeros(X.shape[1])
    losses = [mse_loss(X, y, w)]
    
    for step in range(n_steps):
        residuals = X @ w - y
        grad = (2 / n) * X.T @ residuals
        w = w - lr * grad
        losses.append(mse_loss(X, y, w))
    
    return w, losses

In [ ]:
w_gd, losses_gd = gradient_descent(X, y, lr=0.1, n_steps=100)

print(f"Gradient Descent (100 steps, lr=0.1):")
print(f"  Final weights: [{w_gd[0]:.4f}, {w_gd[1]:.4f}]")
print(f"  True weights:  [{w_true[0]:.4f}, {w_true[1]:.4f}]")
print(f"  Final loss:    {losses_gd[-1]:.4f}")

## Newton's Method

Instead of $\Delta \mathbf{w} = -\eta \nabla L$, Newton's method uses $\Delta \mathbf{w} = -H^{-1} \nabla L$.

The Hessian inverse adjusts the gradient to account for cross-parameter interactions. For linear regression, the Hessian is constant, so Newton's method should reach the exact minimum in a single step.

In [ ]:
def newtons_method(X, y, n_steps=5):
    n = X.shape[0]
    w = np.zeros(X.shape[1])
    losses = [mse_loss(X, y, w)]
    
    H = (2 / n) * X.T @ X
    H_inv = np.linalg.inv(H)
    
    for step in range(n_steps):
        residuals = X @ w - y
        grad = (2 / n) * X.T @ residuals
        w = w - H_inv @ grad
        losses.append(mse_loss(X, y, w))
    
    return w, losses

In [ ]:
w_newton, losses_newton = newtons_method(X, y, n_steps=5)

print(f"Newton's Method (5 steps):")
print(f"  Final weights: [{w_newton[0]:.4f}, {w_newton[1]:.4f}]")
print(f"  True weights:  [{w_true[0]:.4f}, {w_true[1]:.4f}]")
print(f"  Final loss:    {losses_newton[-1]:.4f}")
print(f"")
print(f"Loss at each step:")
for i, loss in enumerate(losses_newton):
    print(f"  Step {i}: {loss:.4f}")

## The Comparison

Newton's method reaches the minimum in 1 step. Gradient descent is still working on it after 100.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(losses_gd, label='Gradient Descent (lr=0.1)', linewidth=2)
plt.plot(range(len(losses_newton)), losses_newton, 'o-', label="Newton's Method", 
         linewidth=2, markersize=10, color='orange')
plt.axhline(y=losses_newton[-1], color='orange', linestyle='--', alpha=0.5, label='Optimal loss')
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('Gradient Descent vs Newton\'s Method')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print(f"After 100 steps of GD:  loss = {losses_gd[-1]:.4f}")
print(f"After 1 step of Newton: loss = {losses_newton[1]:.4f}")
print(f"Gap: {losses_gd[-1] - losses_newton[1]:.4f}")

## Why the Gap?

Gradient descent computes each `.grad` assuming the other weight is constant. But the features are correlated — the weights are entangled. So each update is slightly wrong, because it doesn't account for the other weight changing at the same time.

Newton's method uses the Hessian's cross-term to correct for this. It knows that when $w_1$ changes, the optimal $w_2$ also shifts — and it adjusts accordingly.

For this 2-parameter problem, the Hessian is a 2x2 matrix. Trivial to compute and invert.

But a model with $N$ parameters has an $N \times N$ Hessian. A million parameters means a trillion entries. $O(N^2)$ memory, $O(N^3)$ to invert. Gradient descent is $O(N)$.

**Gradient descent wins not because it's right, but because it's fast enough to iterate.**